In [1]:
import boto3

In [2]:
session = boto3.Session(
    region_name='us-east-1',
    aws_access_key_id='test',
    aws_secret_access_key='test'
)

In [3]:
endpoint='http://localhost:4566'
kinesis = session.client('kinesis', endpoint_url=endpoint)

In [6]:
shards = kinesis.list_shards(StreamName='pr-events-stream')

In [7]:
shards

{'Shards': [{'ShardId': 'shardId-000000000000',
   'HashKeyRange': {'StartingHashKey': '0',
    'EndingHashKey': '340282366920938463463374607431768211455'},
   'SequenceNumberRange': {'StartingSequenceNumber': '0',
    'EndingSequenceNumber': '1795296782364'}},
  {'ShardId': 'shardId-000000000001',
   'ParentShardId': 'shardId-000000000000',
   'HashKeyRange': {'StartingHashKey': '0',
    'EndingHashKey': '170141183460469231731687303715884105727'},
   'SequenceNumberRange': {'StartingSequenceNumber': '1795296782364'}},
  {'ShardId': 'shardId-000000000002',
   'ParentShardId': 'shardId-000000000000',
   'HashKeyRange': {'StartingHashKey': '170141183460469231731687303715884105728',
    'EndingHashKey': '340282366920938463463374607431768211455'},
   'SequenceNumberRange': {'StartingSequenceNumber': '1795296782364'}}],
 'ResponseMetadata': {'RequestId': 'f9cbcb39-2ea7-48eb-92ac-296bbeb0adea',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'content-type': 'application/x-amz-json-1.1;charset=UTF

In [62]:
next_shard_iterator = {}
for shard in shards['Shards']:
    shard_iterator = kinesis.get_shard_iterator(StreamName='pr-events-stream', ShardId=shard['ShardId'], ShardIteratorType='TRIM_HORIZON')
    records = kinesis.get_records(ShardIterator=shard_iterator['ShardIterator'], Limit=25)
    print(f'ShardId: {shard['ShardId']}: {len(records['Records'])}')
    if len(records['Records']) > 0:
        next_shard_iterator[shard['ShardId']] = records['NextShardIterator']

ShardId: shardId-000000000000: 0
ShardId: shardId-000000000001: 5
ShardId: shardId-000000000002: 17


In [63]:
next_shard_iterator

{'shardId-000000000001': 'cHItZXZlbnRzLXN0cmVhbXxzaGFyZElkLTAwMDAwMDAwMDAwMXxBRlRFUl9TRVFVRU5DRV9OVU1CRVJ8MTc5NTI5Njc4MjM3NnwwfA==',
 'shardId-000000000002': 'cHItZXZlbnRzLXN0cmVhbXxzaGFyZElkLTAwMDAwMDAwMDAwMnxBRlRFUl9TRVFVRU5DRV9OVU1CRVJ8MTc5NTI5Njc4MjM4NnwwfA=='}

In [52]:
# test with ShardIteratorType = 'LATEST'
latest_iterators = {}
for shard in shards['Shards']:
    sid = shard['ShardId']
    resp = kinesis.get_shard_iterator(
        StreamName='pr-events-stream',
        ShardId=sid,
        ShardIteratorType='LATEST',
    )
    latest_iterators[sid] = resp['ShardIterator']

In [55]:
for sid, iterator in latest_iterators.items():
    records = kinesis.get_records(ShardIterator=iterator, Limit=25)
    print(f"ShardId: {sid}: {len(records['Records'])}")
    # Advance the pointer for the next poll
    latest_iterators[sid] = records['NextShardIterator']

ShardId: shardId-000000000000: 0
ShardId: shardId-000000000001: 0
ShardId: shardId-000000000002: 2


In [56]:
latest_iterators

{'shardId-000000000000': 'cHItZXZlbnRzLXN0cmVhbXxzaGFyZElkLTAwMDAwMDAwMDAwMHxUUklNX0hPUklaT058fDB8',
 'shardId-000000000001': 'cHItZXZlbnRzLXN0cmVhbXxzaGFyZElkLTAwMDAwMDAwMDAwMXxBRlRFUl9TRVFVRU5DRV9OVU1CRVJ8MTc5NTI5Njc4MjM3NnwwfA==',
 'shardId-000000000002': 'cHItZXZlbnRzLXN0cmVhbXxzaGFyZElkLTAwMDAwMDAwMDAwMnxBRlRFUl9TRVFVRU5DRV9OVU1CRVJ8MTc5NTI5Njc4MjM4MnwwfA=='}

In [61]:
for sid, iterator in latest_iterators.items():
    records = kinesis.get_records(ShardIterator=iterator, Limit=25)
    print(f"ShardId: {sid}: {len(records['Records'])}")
    # Advance the pointer for the next poll
    latest_iterators[sid] = records['NextShardIterator']

ShardId: shardId-000000000000: 0
ShardId: shardId-000000000001: 0
ShardId: shardId-000000000002: 0
